# Generalization to multi-batch 

Now that we can safely do things with $n=1$, let's do a batch and be able to fit/train things in 1D at least. 

We are primarily interested in making the constraint's total derivative matrices first in an explicit way before moving to the matrix-free versions. 

## Imports and model definition

We use a residual structure and typical Jax construction. The main difference is that the naming of the layers is changed for lexcicographic sorting. 

In [1]:
import jax
jax.config.update("jax_enable_x64", True)
jax.config.update('jax_default_matmul_precision', 'highest')
import jax.numpy as jnp
from flax import linen as nn
import numpy as np

# For viz
import treescope
treescope.basic_interactive_setup(
    autovisualize_arrays=True,
    abbreviation_threshold=2, 
)
treescope.register_as_default()


In [2]:
class MLP(nn.Module):
    """
    We first deal with something that's not exactly MLP, but close enough
    """
    num_units: int
    
    def setup(self):
        self.dense1 = nn.Dense(self.num_units)
        self.dense2 = nn.Dense(self.num_units)
    
    def __call__(self, x):
        f = self.dense1(x)
        # f = nn.leaky_relu(f)
        f = nn.tanh(f)
        f = self.dense2(f)
        # f = nn.tanh(f)
        # x = self.dense2(x)
        return x + f
    

class SimpleMLP(nn.Module):
    num_layers: int
    num_units: int
    num_classes: int

    def setup(self):
        # Create a list of Dense layers
        self.layers = [
            MLP(self.num_units, name=f"layer_{i:02}") for i in range(self.num_layers)
        ]
        
    def __call__(self, x):
        # Store input to match the notes
        self.sow('intermediates',  f'layer_{0}_output', x)
        
        # Pass the input through each layer
        for i, layer in enumerate(self.layers):
            x = layer(x)
            
            # Store input to match the notes
            self.sow('intermediates', f'layer_{i+1}_output', x)
            
        # Final layer to produce output
        # x = self.classification_layer(x)
        return x


## Construction

Of the dg/dt and dg/du

Using these explicit constructions will run into memory problems very fast, but preconditioner can be done locally and maybe in matrix free way? 

In [3]:
n = 1 # 1 dimensional input and output 
L = 9 # Number of layers; trying to emulate the paper
n_samples = 2
x = jax.random.normal(jax.random.PRNGKey(0), (n_samples, n))

# First, pretend to call the model
model = SimpleMLP(num_layers=L, num_units=n, num_classes=n)
params = model.init( jax.random.PRNGKey(0), x)
predictions, intermediates = model.apply(params, x, mutable=['intermediates'])
intermediates

{'intermediates': {'layer_0_output': (<jax.Array float64(2, 1) ≈-0.5 ±0.29 [≥-0.78, ≤-0.21] nonzero:2
     <Arrayviz rendering>
   | Device: GPU 0>,),
  'layer_1_output': (<jax.Array float64(2, 1) ≈-0.91 ±0.48 [≥-1.4, ≤-0.43] nonzero:2
     <Arrayviz rendering>
   | Device: GPU 0>,),
  'layer_2_output': (<jax.Array float64(2, 1) ≈-0.58 ±0.32 [≥-0.9, ≤-0.26] nonzero:2
     <Arrayviz rendering>
   | Device: GPU 0>,),
  'layer_3_output': (<jax.Array float64(2, 1) ≈-0.72 ±0.36 [≥-1.1, ≤-0.35] nonzero:2
     <Arrayviz rendering>
   | Device: GPU 0>,),
  'layer_4_output': (<jax.Array float64(2, 1) ≈-2.2 ±0.81 [≥-3.0, ≤-1.4] nonzero:2
     <Arrayviz rendering>
   | Device: GPU 0>,),
  'layer_5_output': (<jax.Array float64(2, 1) ≈-1.3 ±0.53 [≥-1.8, ≤-0.73] nonzero:2
     <Arrayviz rendering>
   | Device: GPU 0>,),
  'layer_6_output': (<jax.Array float64(2, 1) ≈-0.011 ±0.34 [≥-0.35, ≤0.32] nonzero:2
     <Arrayviz rendering>
   | Device: GPU 0>,),
  'layer_7_output': (<jax.Array float64(2, 1) ≈-0.014 ±0.45 [≥-0.46, ≤0.44] nonzero:2
     <Arrayviz rendering>
   | Device: GPU 0>,),
  'layer_8_output': (<jax.Array float64(2, 1) ≈-0.014 ±0.45 [≥-0.46, ≤0.43] nonzero:2
     <Arrayviz rendering>
   | Device: GPU 0>,),
  'layer_9_output': (<jax.Array float64(2, 1) ≈-0.024 ±0.82 [≥-0.85, ≤0.8] nonzero:2
     <Arrayviz rendering>
   | Device: GPU 0>,)}}

Need derivative of layers wrt to both inputs and weights. 

$K_i$ is the gradient wrt to parameters while $M_i$ is gradient wrt to $x$s. 

In [17]:
# Create layer model; then be able to take the gradients and stuff
layer = MLP(num_units=n)

# First define function
def apply_layer(params, x): 
    assert len(x) == 1
    return jnp.squeeze(layer.apply(params, x))

# The issue with jax.grad is it doesn't automatically vectorize over batch dimension, so we do it manually with vmap and squeeze
K_i_one = jax.grad(apply_layer, 0)
K_i = jax.vmap(K_i_one, (None, 0)) # We don't allow vmapping over the params. I guess we could in theory... but it's easier to wrap my head around htis

M_i_one = jax.grad(apply_layer, 1)
M_i = jax.vmap(M_i_one, (None, 0))



In [18]:
dgdu = np.eye(n_samples * (L + 1)) # It's tridiagonal with ones down diagonal; will have to change once scaled up
counter = 0
for l in range(L):
    # print(-jnp.squeeze(M_i({'params': params['params'][f'layer_{l:02}']}, intermediates['intermediates'][f'layer_{l}_output'][0])))
    # print(dgdu[l, l+1:l+n_samples+1] )
    vals = -jnp.squeeze(M_i({'params': params['params'][f'layer_{l:02}']}, intermediates['intermediates'][f'layer_{l}_output'][0]))
    # TODO: don't make this a for loop eventually 
    for s in range(n_samples):
        dgdu[counter * n_samples + s, counter * n_samples + n_samples + s] =  vals[s]
    counter += 1
dgdu.T

array([[ 1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  1.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [-2.01103882,  0.        ,  1.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        , -1.34948179,  0.        ,  1.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        , -0.61517394,  0.        ,  1.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , -0.71305597,  0.        ,
         1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        , -1.30500538,
         0.        ,  1.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        -1.04092156,  0.        ,  1.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        , -3.41674201,  0.        ,  1.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        , -1.40716669,  0.        ,  1.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        , -0.57065812,  0.        ,
         1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        , -0.74909008,
         0.        ,  1.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        -0.09670041,  0.        ,  1.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        , -0.90916429,  0.        

In [6]:
# Number of trainable parameters
p = sum(x.size for x in jax.tree.leaves(params))

dgdt = np.zeros((p, n_samples * (L + 1)))
# This is harder to construct; need a mapping from dof to matrix... think about this for bigger problems
# Better way would be a matrix free operation; but for now, just do 
counter = 0
for l in range(L): 
    # print(K_i({'params': params['params'][f'layer_{l:02}']}, intermediates['intermediates'][f'layer_{l}_output'][0]))
    flattened, _ = jax.tree.flatten(
            K_i({'params': params['params'][f'layer_{l:02}']}, intermediates['intermediates'][f'layer_{l}_output'][0])
    )
    
    vals = np.array([jnp.squeeze(x) for x in flattened])
    layer_p = len(vals)
    # print(vals)

    # Can make this dynamic 
    dgdt[counter:counter + layer_p, n_samples * l + n_samples:n_samples * (l + 1) + n_samples] = vals

    counter += layer_p
dgdt


array([[ 0.00000000e+00,  0.00000000e+00, -6.71473086e-01,
        -2.32105434e-01,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  1.38217449e-01,
         1.82148397e-01,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  1.00000000e+00,
         1.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  3.00379843e-01,
         8.27962995e-01,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  9.16369677e-01,  6.83287382e-01,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00, -3.91787171e-01, -9.53772426e-01,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  1.00000000e+00,  1.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  1.77640259e-01,  5.27146697e-01,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        -1.51969373e-01, -2.03892272e-02,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         3.94291207e-02,  1.82902962e-02,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00

In [7]:
derivative = np.linalg.inv(dgdu.T) @ dgdt.T
np.linalg.inv(dgdu.T)
derivative

array([[-7.41392829e-17,  1.52609878e-17,  1.10412888e-16,
         3.31658060e-17,  1.64472219e-16, -7.03188977e-17,
         1.79482390e-16,  3.18832982e-17, -2.09009301e-17,
         5.42283802e-18,  1.37533831e-16,  6.57773535e-17,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-6.71473086e-01,  1.38217449e-01,  1.00000000e+00,
         3.00379843e-01,  3.30760018e-16, -1.41414033e-16,
         3.60946053e-16,  6.41185505e-17, -4.20325818e-17,
         1.09055378e-17,  2.76585873e-16,  1.32280811e-16,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-2.32105434e-01,  1.82148397e-01,  1.00000000e+00,
         8.27962995e-01,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-4.13072742e-01,  8.50277725e-02,  6.15173938e-01,
         1.84785851e-01,  9.16369677e-01, -3.91787171e-01,
         1.00000000e+00,  1.77640259e-01, -2.58573488e-17,
         6.70880262e-18,  1.70148421e-16,  8.13757077e-17,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-1.65504166e-01,  1.29882002e-01,  7.13055973e-01,
         5.90383958e-01,  6.83287382e-01, -9.53772426e-01,
         1.00000000e+00,  5.27146697e-01, -4.34933626e-18,
         3.90160194e-18,  2.13315405e-16,  2.01977361e-16,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00, 

In [8]:
# Let's compare this with the Jacobian 
jacobian = jax.jacobian(model.apply, argnums=0)(params, x)
jacobian = jnp.array([jnp.squeeze(x) for x in jax.tree.flatten(jacobian)[0]])
jacobian.T 

<jax.Array float32(2, 36) ≈0.32 ±1.2 [≥-2.1, ≤5.1] nonzero:72
  <Arrayviz rendering>
| Device: GPU 0>

In [9]:
np.linalg.norm(derivative[-n_samples:, :] - jacobian.T ) / np.linalg.norm(jacobian)

np.float64(4.127935584722667e-08)

## Construction of precond 

The preconditioner shouldn't change much

In [19]:
# Matrix which we want ot approximate 
jnp.linalg.inv(dgdu.T @ dgdu)

<jax.Array float64(20, 20) ≈3.1 ±5.8 [≥0.0, ≤5e+01] zero:200 nonzero:200
  <Arrayviz rendering>
| Device: GPU 0>

In [10]:
def partition_with_overlap(L, N):
    # Compute the approximate size of each partition
    step = L / N
    
    # Generate the partition points
    c = [round(i * step) for i in range(N + 1)]
    
    return c


In [11]:
def make_matrices(L, ND): 
    """
    Given L layers and ND blocks, give the restriction, pou and q matrices for testing
    """
    restrictions = []
    pous = []
    qs = []

    cs = partition_with_overlap(L, ND)
    for d in range(ND): 
        time_steps = cs[d + 1] - cs[d] + 1
        r = np.zeros((n_samples * time_steps, n_samples * (L + 1)))
        for t in range(n_samples * time_steps):
            r[t, n_samples * cs[d] + t] = 1

        pou = np.eye(n_samples * time_steps)
        if d == 0:
            pou[np.arange(len(pou) - n_samples, len(pou)), np.arange(len(pou) - n_samples, len(pou))] = 0.5
        elif d == ND - 1:
            pou[np.arange(0, n_samples), np.arange(0, n_samples)] = 0.5
        else: 
            pou[np.arange(0, n_samples), np.arange(0, n_samples)] = 0.5
            pou[np.arange(len(pou) - n_samples, len(pou)), np.arange(len(pou) - n_samples, len(pou))] = 0.5

        if d == 0:
            q = np.zeros((n_samples * time_steps, n_samples * (L + 1)))
            for i in range(n_samples * time_steps): 
                q[i, i] = 1
        else:
            q = np.zeros((n_samples * (time_steps + 1), n_samples *( L + 1)))
            for t in range(n_samples * (time_steps + 1)): 
                q[t, n_samples * cs[d] - n_samples + t] = 1
        
        restrictions.append(r)
        pous.append(pou)
        qs.append(q)

    # assert np.linalg.norm(R1.T @ D1 @ R1 + R2.T @ D2 @ R2 + R3.T @ D3 @ R3 - np.eye(10)) < 1e-10

    return restrictions, pous, qs


In [12]:
ND = 3
make_matrices(L, ND)

([array([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.]]),
  array([[0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
          0., 0., 0., 0.]]),
  array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          1., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 1., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 1., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 1.]])],
 [array([[1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
         [0. , 1. , 0. , 0. , 0. , 0. , 0. , 0. ],
         [0. , 0. , 1. , 0. , 0. , 0. , 0. , 0. ],
         [0. , 0. , 0. , 1. , 0. , 0. , 0. , 0. ],
         [0. , 0. , 0. , 0. , 1. , 0. , 0. , 0. ],
         [0. , 0. , 0. , 0. , 0. , 1. , 0. , 0. ],
         [0. , 0. , 0. , 0. , 0. , 0. , 0.5, 0. ],
         [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5]]),
  array([[0.5, 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
         [0. , 0.5, 0. , 0. , 0. , 0. , 0. , 0. ],
         [0. , 0. , 1. , 0. , 0. , 0. , 0. , 0. ],
         [0. , 0. , 0. , 1. , 0. , 0. , 0. , 0. ],
         [0. , 0. , 0. , 0. , 1. , 0. , 0. , 0. ],
         [0. , 0. , 0. , 0. , 0. , 1. , 0. , 0. ],
         [0. , 0. , 0. , 0. , 0. , 0. , 0.5, 0. ],
         [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5]]),
  array([[0.5, 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
         [0. , 0.5, 0. , 0. , 0. , 0. , 0. , 0. ],
         [0. , 0. , 1. , 0. , 0. , 0. , 0. , 0. ],
         [0. , 0. , 0. , 1. , 0. , 0. , 0. , 0. ],
         [0. , 0. , 0. , 0. , 1. , 0. , 0. , 0. ],
         [0. , 0. , 0. , 0. , 0. , 1. , 0. , 0. ],
         [0. , 0. , 0. , 0. , 0. , 0. , 1. , 0. ],
         [0. , 0. , 0. , 0. , 0. , 0. , 0. , 1. ]])],
 [array([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 1., 0., 0., 0

In [13]:
restrictions, pous, qs = make_matrices(L, ND)
# Construct RAS preconditioner 
ras = np.zeros((n_samples * (L + 1), n_samples * (L + 1)))
for i in range(ND): 
    R = restrictions[i]
    D = pous[i]
    ras += R.T @ D @ jnp.linalg.inv(R @ dgdu.T @ dgdu @ R.T) @ R
    # print(i)


In [14]:
ras

<jax.Array float64(20, 20) ≈0.32 ±0.96 [≥0.0, ≤9.2] zero:308 nonzero:92
  <Arrayviz rendering>
| Device: GPU 0>

In [15]:
    print(np.linalg.eigvals(ras @  dgdu.T @ dgdu))


[0.04660735+0.00000000e+00j 0.76588933+0.00000000e+00j
 1.23411067+0.00000000e+00j 1.95339265+0.00000000e+00j
 0.04928929+0.00000000e+00j 1.95071071+0.00000000e+00j
 0.45992038+0.00000000e+00j 1.54007962+0.00000000e+00j
 1.        +0.00000000e+00j 1.        +0.00000000e+00j
 1.        +0.00000000e+00j 1.        +0.00000000e+00j
 1.        +0.00000000e+00j 1.        +0.00000000e+00j
 1.        +0.00000000e+00j 1.        +0.00000000e+00j
 1.        +0.00000000e+00j 1.        +0.00000000e+00j
 1.        +5.19259273e-17j 1.        -5.19259273e-17j]


In [20]:
rasq = np.zeros((n_samples * (L + 1), n_samples * (L + 1)))
for i in range(ND): 
    R = restrictions[i]
    D = pous[i]
    Q = qs[i]
    Js = np.linalg.inv(Q @ dgdu @ Q.T)
    rasq += R.T @ D @ (R @ Q.T @ Js @ Js.T @ Q @ R.T) @ R
print(np.linalg.eigvals(rasq @  dgdu.T @ dgdu))
rasq

[15.94478747+0.00000000e+00j  0.11376382+0.00000000e+00j
  1.16640995+0.00000000e+00j  6.17192651+0.00000000e+00j
  0.87918445+0.00000000e+00j  3.41479168+0.00000000e+00j
  0.121243  +0.00000000e+00j  0.85271922+0.00000000e+00j
  1.        +7.28021923e-16j  1.        -7.28021923e-16j
  1.        +1.33688556e-15j  1.        -1.33688556e-15j
  1.        +0.00000000e+00j  1.        +0.00000000e+00j
  1.        +0.00000000e+00j  1.        +0.00000000e+00j
  1.        +0.00000000e+00j  1.        +0.00000000e+00j
  1.        +0.00000000e+00j  1.        +0.00000000e+00j]


array([[9.18131109, 0.        , 4.06820147, 0.        , 3.34403414,
        0.        , 1.61447263, 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 4.75030661, 0.        , 2.77907167, 0.        ,
        2.00487751, 0.        , 1.00163307, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [4.06820147, 0.        , 2.02293533, 0.        , 1.66283918,
        0.        , 0.8028053 , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 2.77907167, 0.        , 2.05936212, 0.        ,
        1.48566474, 0.        , 0.74223534, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [3.34403414, 0.        , 1.66283918, 0.        , 2.70303905,
        0.        , 1.30500538, 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 2.00487751, 0.        , 1.48566474, 0.        ,
        2.0835177 , 0.        , 1.04092156, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.80723631, 0.        , 0.40140265, 0.        , 0.65250269,
        0.        , 8.75568129, 0.        , 2.26990544, 0.        ,
        0.984012  , 0.        , 0.09427282, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.50081654, 0.        , 0.37111767, 0.        ,
        0.52046078, 0.        , 3.00482951, 0.        , 1.42472781,
        0.        , 0.96269391, 0.        , 0.47917259, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 4.53981089, 0.        , 1.32869584, 0.        ,
        0.57599433, 0.        , 0.05518288, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 2.84945561, 0.        , 2.02495954,
        0.        , 1.36827274, 0.        , 0.68104595, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 1.96802401, 0.        , 0.57599433, 0.        ,
        1.00935097, 0.        , 0.09670041, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 1.92538783, 0.        , 1.36827274,
        0.        , 1.82657971, 0.        , 0.90916429, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.09427282, 0.        , 0.02759144, 0.        ,
        0.04835021, 0.        , 5.24796368, 0.        , 3.20152702,
        0.        , 2.53947092, 0.        , 1.11543016, 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.47917259, 0.        , 0.34052298,
        0.        , 0.45458215, 0.        , 5.15369917, 0.        ,
        3.13739817, 0.        , 2.47677037, 0.        , 1.09586768],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 6.40305404

## Need to do multi-dimensional
